# SRΨ-Engine Ablation Study on Google Colab

This notebook runs the complete ablation study on Google Colab GPU.

## Experiments
- Exp1: SRΨ Full (Already completed)
- Exp2: SRΨ Real-only (No complex state)
- Exp3: SRΨ w/o R (No Rhythm operator)
- Exp4: Conv Baseline (Pure convolution)
- Exp5: Transformer Rel-PE (Relative position encoding)

## Expected Time
- Each experiment: ~30-40 minutes on GPU
- Total: ~2.5-3 hours (or parallel with multiple tabs)

## 1. Check GPU Availability

In [ ]:
# Check if GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ No GPU detected. Click Runtime → Change runtime type → GPU')

## 2. Clone Repository

In [ ]:
# Clone the repository
!git clone https://github.com/Mozilla2004/srpsi-engine-tiny.git
%cd srpsi-engine-tiny

print('✓ Repository cloned')
!ls -la

## 3. Install Dependencies

In [ ]:
# Install required packages
!pip install -q pyyaml tqdm

import torch
import yaml
print('✓ Dependencies installed')
print(f'✓ PyTorch {torch.__version__} ready')

## 4. Download Training Data

In [ ]:
# Check if data exists
import os

if os.path.exists('data/burgers_1d.npy'):
    size_mb = os.path.getsize('data/burgers_1d.npy') / (1024*1024)
    print(f'✓ Data already exists: {size_mb:.1f} MB')
else:
    print('Generating training data...')
    !python src/data_gen.py
    
    if os.path.exists('data/burgers_1d.npy'):
        print('✓ Data generated successfully')
    else:
        print('✗ Data generation failed')

## 5. Run Experiment

Choose which experiment to run by modifying the `model_name` variable below.

In [ ]:
# Select experiment to run
model_name = 'srpsi_real'  # Options: srpsi_real, srpsi_no_r, conv_baseline, transformer_rel_pe

print(f'Starting experiment: {model_name}')
print('='*60)

# Run training
!python src/train.py \
  --config config/burgers.yaml \
  --model {model_name} \
  --data data/burgers_1d.npy \
  --output outputs/ablation_{model_name} \
  --epochs 80

## 6. Download Results

After training completes, download the best model checkpoint.

In [ ]:
# Find and display best checkpoint
import os
from glob import glob

checkpoint_dir = f'outputs/ablation_{model_name}/{model_name}/checkpoints'
checkpoints = glob(f'{checkpoint_dir}/best*.pt')

if checkpoints:
    best_model = checkpoints[0]
    size_mb = os.path.getsize(best_model) / (1024*1024)
    print(f'✓ Best model: {best_model}')
    print(f'  Size: {size_mb:.1f} MB')
    print('\nTo download, right-click on the file in the left panel and select "Download"')
else:
    print('⚠️ No best checkpoint found. Training may still be in progress.')

## 7. Monitor Training Progress

In [ ]:
# Check training logs
!tail -50 logs/ablation_{model_name}.log